# Construction of the Contract-Year dataset

This notebook prepares the contract metadata for the thesis analysis by transforming the raw contracts table into a **contract-year dataset**.

The original dataset contains **one row per contract**, including contract identifiers, supplier information, lifecycle dates, and commercial attributes. However, several supporting data sources used later in the analysis (e.g., financial data, spend data, and ESG indicators) are structured on a **yearly basis**.

To enable consistent integration across these sources, contracts are expanded into a **contract-year representation**, where `each row corresponds to a contract observed in a specific year during its active lifetime`.

The resulting `contract-year dataset` serves as the structural foundation for the subsequent integration of supplier, financial, and spend data used in the modeling framework.

In [35]:
# import libraries
import pandas as pd
import numpy as np

## Contract metadata

In [36]:
# CI Metadata
df_contracts = pd.read_csv(
    "/Users/Thomas/Desktop/Master Thesis/Data/dim_ci_stso_contracts_metadata.csv",
    engine="python",
    on_bad_lines="skip"
)

In [37]:
df_contracts.shape

(2252, 97)

In [38]:

df_contracts.columns.tolist()

['contract_id',
 'contract_number',
 'contract_name',
 'contract_status',
 'terminated',
 'term_type',
 'contract_owner',
 'contract_owner_cost_centre',
 'start_date',
 'expiration_date',
 'created_by',
 'created_at',
 'updated_by',
 'updated_at',
 'submitted_by',
 'submitted_at',
 'published_at',
 'execution_at',
 'supplier_id',
 'supplier_number',
 'version',
 'contract_type',
 'nn_contract_type',
 'parent_contract_number',
 'parent_contract_name',
 'master_contract_number',
 'master_contract_name',
 'amended_contract_type',
 'content_groups',
 'description',
 'contract_tags',
 'payment_terms',
 'incoterms',
 'savings_pct',
 'contract_currency_code',
 'contract_value',
 'minimum_spend',
 'minimum_value',
 'novo_nordisk_compliance_requirements',
 'novo_nordisk_legal_entity',
 'site_number',
 'approval_and_signing_process',
 'reason_for_approving_and_signing_offline',
 'created_from_id',
 'created_from_type',
 'hierarchy_type',
 'quote_response_id',
 'linked_events',
 'linked_projects'

In [39]:
df_contracts.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,contract_owner,contract_owner_cost_centre,start_date,expiration_date,...,legal_agreement_file_name,is_the_contract_gxp_regulated,attachments_names,supplier_display_name,Preferred_supplier_tag,unit,department,team,days_until_expiry,expiry_range
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,RSRV,7756.0,2018-05-21T00:00:00.000Z,2027-07-30T00:00:00.000Z,...,NaN,Yes,Bioreliance_MSA_2018.pdf,BIORELIANCE LIMITED INVITROGEN BIOSERVICES,0.0,SSIMS,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",516.0,Over 360 days
1,8157,8158,Kalundborg Bioenergi_Master_2017_SA,published,False,fixed,BLHN,1751.0,2018-04-01T00:00:00.000Z,2033-07-01T00:00:00.000Z,...,NaN,NaN,Kal_Bioenergi_Purchase_Agreement_2017.03.16.pdf,KALUNDBORG BIOENERGI APS,0.0,SSIMS,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",2679.0,Over 360 days
2,1276,1276,Lonza Ltd & Lonza Sales Ltd_Master_2017_FA,published,False,auto_renew,MKYR,1751.0,2017-02-22T00:00:00.000Z,2037-02-22T00:00:00.000Z,...,Lonza_Ltd___Lonza_Sales_Ltd_Master_20170222_FA...,NaN,Lonza_Ltd___Lonza_Sales_Ltd_Internal_doc_20170...,LONZA SALES LTD,0.0,NaN,NaN,NaN,4011.0,Over 360 days
3,201,201,Evonik_Master_20160520_SA,published,False,fixed,PALD,1751.0,2016-05-01T00:00:00.000Z,2026-05-01T00:00:00.000Z,...,Evonik_Master_20160520_SA.pdf,NaN,Evonik_Internal_Doc_20160520_CL.pdf;Evonik_Cal...,"EVONIK REXIM (NANNING) PHARMACEUTIC CO., LTD",0.0,NaN,NaN,NaN,61.0,90 days
4,19779,19779,DONG ENERGY THERMAL POWER A/S_Master_2017_SA_S...,published,False,fixed,ENOM,1751.0,2017-06-22T00:00:00.000Z,2039-12-31T00:00:00.000Z,...,DONG_Energy_Thermal_Power_A_S_SA_20170623.PDF,NaN,DONG_Energy_Thermal_Power_A_S_SA_APP_20170623_...,DONG ENERGY THERMAL POWER A/S,0.0,SSIMS,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",5053.0,Over 360 days


Do we have one row per contract & no duplicates?

- Yes!

contract_id == contract_number

In [40]:
df_contracts["contract_id"].nunique(), len(df_contracts)

(2252, 2252)

In [41]:
df_contracts["contract_number"].nunique(), len(df_contracts)

(2252, 2252)

In [42]:
df_contracts[df_contracts["contract_id"].duplicated(keep=False)].sort_values("contract_id")

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,contract_owner,contract_owner_cost_centre,start_date,expiration_date,...,legal_agreement_file_name,is_the_contract_gxp_regulated,attachments_names,supplier_display_name,Preferred_supplier_tag,unit,department,team,days_until_expiry,expiry_range


Let's check the date structure now

In [43]:
# convert date columns
df_contracts["start_date"] = pd.to_datetime(df_contracts["start_date"], errors="coerce")
df_contracts["expiration_date"] = pd.to_datetime(df_contracts["expiration_date"], errors="coerce")

# check range
print(df_contracts["start_date"].min())
print(df_contracts["start_date"].max())

print(df_contracts["expiration_date"].min())
print(df_contracts["expiration_date"].max())

# check missing
print(df_contracts[["start_date","expiration_date"]].isna().mean())

2001-01-10 00:00:00+00:00
2026-03-01 00:00:00+00:00
2026-03-01 00:00:00+00:00
2099-12-31 00:00:00+00:00
start_date         0.000444
expiration_date    0.000444
dtype: float64


Contracts start between: 2001 → 2026

Contracts expire between: 2026 → 2099


In [44]:
df_contracts["contract_duration_years"] = (
    df_contracts["expiration_date"] - df_contracts["start_date"]
).dt.days / 365

df_contracts["contract_duration_years"].describe()

count    2251.000000
mean       53.393043
std        35.852108
min         0.243836
25%         5.000000
50%        76.134247
75%        79.049315
max        99.038356
Name: contract_duration_years, dtype: float64

## Building the Contract-Year dataset

### Goal
The original contracts dataset is at **contract level**, meaning that each row represents one contract.  
However, the later modeling pipeline requires a **contract-year structure**, so that time-varying datasets such as spend, supplier financials, and ESG can be joined correctly.

The target structure is:

**`one row = one contract in one year`**


### Why this transformation is needed
The raw contracts table contains one row per contract, for example:

| contract_id | start_date | expiration_date |
|---|---|---|
| 9675 | 2018-05-21 | 2027-07-30 |

This format is not sufficient for joining yearly supplier data such as:

| supplier_id | year | revenue |
|---|---|---|
| S1 | 2018 | ... |
| S1 | 2019 | ... |

To connect contracts to these yearly sources, each contract must be expanded into the years during which it is active.

### Step 1: Parse the date fields
The columns `start_date` and `expiration_date` are converted from text to datetime format.  
These two fields define the active period of the contract.

### Step 2: Extract start and expiry years
For each contract, we derive:

- `start_year`
- `expiry_year`

`These fields are used to determine the yearly observation window.`

### Step 3: Define the analysis window
Some contracts have far-future expiration dates such as `2099-12-31`, which usually indicate `open-ended` or framework agreements rather than true 80-year durations.

To avoid expanding these contracts unrealistically, an analysis cap is applied:

- `analysis_year_min = 2015`
- `analysis_year_max = 2025`

This means that contracts are only represented for years within the relevant modeling horizon.

### Step 4: Expand each contract into multiple yearly rows
`Each contract is transformed from one row into multiple rows, one for each active year.`

Example:

Original row:

| contract_id | start_year | expiry_year |
|---|---|---|
| 9675 | 2018 | 2027 |

After applying `analysis_year_max = 2025`, the contract is expanded to:

| contract_id | year |
|---|---|
| 9675 | 2018 |
| 9675 | 2019 |
| 9675 | 2020 |
| 9675 | 2021 |
| 9675 | 2022 |
| 9675 | 2023 |
| 9675 | 2024 |
| 9675 | 2025 |

### Step 5: Create time-dependent contract features
After expansion, additional yearly features can be created, such as:

- `years_to_expiry = expiry_year - year`
- `contract_age_years = year - start_year`

These are useful for later renegotiation signals, since a contract may become more relevant as it approaches expiry.

## Output
The result of this transformation is a new dataset called the **contract-year backbone**, where:

- the original contract attributes are repeated across years
- each row represents one contract-year observation
- later yearly sources can be joined on `supplier_id + year`

This table becomes the base for the later steps:
1. supplier enrichment
2. Moody’s financial features
3. spend aggregation
4. weak labeling with Snorkel
5. base learner training
6. meta-learning

### Step 1: Parse date columns

The relevant contract lifecycle date fields are converted to datetime format so they can be used to derive yearly contract boundaries.

In [45]:
# Parse relevant date columns
date_cols = ["start_date", "expiration_date", "execution_at", "published_at"]
for col in date_cols:
    if col in df_contracts.columns:
        df_contracts[col] = pd.to_datetime(df_contracts[col], errors="coerce")

### Step 2: Create the contract-level working table

A reduced contract table is created by keeping only the columns needed for the contract-year transformation and later data integration.

In [46]:
# Keep only the columns we need for the contract backbone for now
contract_base = df_contracts[
    [
        "contract_id",
        "contract_number",
        "contract_name",
        "contract_status",
        "terminated",
        "term_type",
        "start_date",
        "expiration_date",
        "supplier_id",
        "supplier_number",
        "supplier_display_name",
        "payment_terms",
        "incoterms",
        "contract_currency_code",
        "contract_value",
        "contract_value_dkk",
        "contract_type",
        "nn_contract_type",
        "contract_commodity",
        "department",
        "team",
        "unit",
        "company_country",
        "days_until_expiry",
        "expiry_range",
        "Preferred_supplier_tag",
    ]
].copy()

In [47]:
contract_base.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,contract_type,nn_contract_type,contract_commodity,department,team,unit,company_country,days_until_expiry,expiry_range,Preferred_supplier_tag
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,Production,Master Service Agreement,Outsourced GMP Laboratory Analysis,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",SSIMS,DENMARK,516.0,Over 360 days,0.0
1,8157,8158,Kalundborg Bioenergi_Master_2017_SA,published,False,fixed,2018-04-01 00:00:00+00:00,2033-07-01 00:00:00+00:00,17030.0,49827,...,Production,NaN,Waste Management,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",SSIMS,DENMARK,2679.0,Over 360 days,0.0
2,1276,1276,Lonza Ltd & Lonza Sales Ltd_Master_2017_FA,published,False,auto_renew,2017-02-22 00:00:00+00:00,2037-02-22 00:00:00+00:00,13689.0,42717,...,Production,NaN,Outsourced Product Services,NaN,NaN,NaN,DENMARK,4011.0,Over 360 days,0.0
3,201,201,Evonik_Master_20160520_SA,published,False,fixed,2016-05-01 00:00:00+00:00,2026-05-01 00:00:00+00:00,16213.0,49030,...,Production,Other,Excipients,NaN,NaN,NaN,DENMARK,61.0,90 days,0.0
4,19779,19779,DONG ENERGY THERMAL POWER A/S_Master_2017_SA_S...,published,False,fixed,2017-06-22 00:00:00+00:00,2039-12-31 00:00:00+00:00,16440.0,46630,...,Production,Master Service Agreement,Fuel,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",SSIMS,DENMARK,5053.0,Over 360 days,0.0


### Step 3: Derive contract start and expiry years

The contract start year and expiry year are extracted from the contract lifecycle dates.  
A flag is also created to identify contracts with far-future expiry dates, which typically represent open-ended agreements.

In [48]:
contract_base["start_year"] = contract_base["start_date"].dt.year
contract_base["expiry_year"] = contract_base["expiration_date"].dt.year

# flag open-ended / placeholder expiries
contract_base["open_ended_contract"] = contract_base["expiry_year"] >= 2090

In [49]:
contract_base.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,department,team,unit,company_country,days_until_expiry,expiry_range,Preferred_supplier_tag,start_year,expiry_year,open_ended_contract
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",SSIMS,DENMARK,516.0,Over 360 days,0.0,2018.0,2027.0,False
1,8157,8158,Kalundborg Bioenergi_Master_2017_SA,published,False,fixed,2018-04-01 00:00:00+00:00,2033-07-01 00:00:00+00:00,17030.0,49827,...,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",SSIMS,DENMARK,2679.0,Over 360 days,0.0,2018.0,2033.0,False
2,1276,1276,Lonza Ltd & Lonza Sales Ltd_Master_2017_FA,published,False,auto_renew,2017-02-22 00:00:00+00:00,2037-02-22 00:00:00+00:00,13689.0,42717,...,NaN,NaN,NaN,DENMARK,4011.0,Over 360 days,0.0,2017.0,2037.0,False
3,201,201,Evonik_Master_20160520_SA,published,False,fixed,2016-05-01 00:00:00+00:00,2026-05-01 00:00:00+00:00,16213.0,49030,...,NaN,NaN,NaN,DENMARK,61.0,90 days,0.0,2016.0,2026.0,False
4,19779,19779,DONG ENERGY THERMAL POWER A/S_Master_2017_SA_S...,published,False,fixed,2017-06-22 00:00:00+00:00,2039-12-31 00:00:00+00:00,16440.0,46630,...,"Quality, Production Services & Supplies","Quality, Production Services & Supplies",SSIMS,DENMARK,5053.0,Over 360 days,0.0,2017.0,2039.0,False


### Step 4: Define the analysis window

To restrict the dataset to the relevant modeling horizon, an analysis window is defined.  
Contracts are only represented for years that fall within this time window.

In [50]:
analysis_year_min = 2015
analysis_year_max = 2025

# Cap end year so we do not expand to 2099
contract_base["end_year"] = contract_base["expiry_year"].clip(upper=analysis_year_max)
# Also cap start year at the lower bound of the analysis window
contract_base["start_year_capped"] = contract_base["start_year"].clip(lower=analysis_year_min)

### Step 5: Keep contracts that overlap the analysis window

Contracts that do not overlap the analysis period are excluded from the working table.

In [51]:
contract_base = contract_base[
    (contract_base["start_year_capped"].notna()) &
    (contract_base["end_year"].notna()) &
    (contract_base["start_year_capped"] <= contract_base["end_year"])
].copy()

### Step 6: Expand contracts into yearly observations

Each contract is expanded into one row per active year, producing the contract-year dataset.

In [52]:
contract_base["year_list"] = contract_base.apply(
    lambda row: list(range(int(row["start_year_capped"]), int(row["end_year"]) + 1)),
    axis=1
)

contract_year = contract_base.explode("year_list").rename(columns={"year_list": "year"}).copy()
contract_year["year"] = contract_year["year"].astype(int)

In [53]:
contract_year.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,company_country,days_until_expiry,expiry_range,Preferred_supplier_tag,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,year
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,DENMARK,516.0,Over 360 days,0.0,2018.0,2027.0,False,2025.0,2018.0,2018
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,DENMARK,516.0,Over 360 days,0.0,2018.0,2027.0,False,2025.0,2018.0,2019
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,DENMARK,516.0,Over 360 days,0.0,2018.0,2027.0,False,2025.0,2018.0,2020
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,DENMARK,516.0,Over 360 days,0.0,2018.0,2027.0,False,2025.0,2018.0,2021
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,DENMARK,516.0,Over 360 days,0.0,2018.0,2027.0,False,2025.0,2018.0,2022


### Step 7: Create time-dependent contract features

Two dynamic features are created for each contract-year observation:

- `years_to_expiry`: remaining years until contract expiry
- `contract_age_years`: number of years since contract start

In [54]:
contract_year["years_to_expiry"] = contract_year["expiry_year"] - contract_year["year"]
contract_year["contract_age_years"] = contract_year["year"] - contract_year["start_year"]

### Step 8: Create an expiry pressure category

A categorical expiry pressure feature is created to group contracts by remaining time to expiry.

In [55]:
contract_year["expiry_pressure_bucket"] = pd.cut(
    contract_year["years_to_expiry"],
    bins=[-np.inf, 0, 1, 3, 5, np.inf],
    labels=["expired_or_expiring", "within_1y", "1_to_3y", "3_to_5y", "5y_plus"]
)

In [56]:
contract_year.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,Preferred_supplier_tag,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,year,years_to_expiry,contract_age_years,expiry_pressure_bucket
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,0.0,2018.0,2027.0,False,2025.0,2018.0,2018,9.0,0.0,5y_plus
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,0.0,2018.0,2027.0,False,2025.0,2018.0,2019,8.0,1.0,5y_plus
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,0.0,2018.0,2027.0,False,2025.0,2018.0,2020,7.0,2.0,5y_plus
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,0.0,2018.0,2027.0,False,2025.0,2018.0,2021,6.0,3.0,5y_plus
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544.0,38367,...,0.0,2018.0,2027.0,False,2025.0,2018.0,2022,5.0,4.0,3_to_5y


### Step 9: Clean identifiers and year fields

The key identifier and year-related columns are converted to consistent types to support reliable filtering and later joins.

In [57]:
contract_year = contract_year.reset_index(drop=True)

contract_year["contract_id"] = contract_year["contract_id"].astype("string")
contract_year["supplier_id"] = contract_year["supplier_id"].astype("Int64")

for col in [
    "year",
    "start_year",
    "expiry_year",
    "end_year",
    "start_year_capped",
    "years_to_expiry",
    "contract_age_years"
]:
    if col in contract_year.columns:
        contract_year[col] = contract_year[col].astype("Int64")

## Step 10: Validate the output

Finally, the dimensions of the original contracts table, the contract-level working table, and the contract-year dataset are compared.  
A small sample of rows is displayed to confirm that the yearly expansion behaves as expected.

In [58]:
print("Original contracts table:", df_contracts.shape)
print("Contract-level table:", contract_base.shape)
print("Contract-year table:", contract_year.shape)

print("\nSample rows:")
print(contract_year[
    [
        "contract_id",
        "contract_name",
        "supplier_id",
        "year",
        "start_year",
        "expiry_year",
        "years_to_expiry",
        "contract_age_years",
        "open_ended_contract"
    ]
].head(15))

Original contracts table: (2252, 98)
Contract-level table: (2209, 32)
Contract-year table: (9201, 35)

Sample rows:
   contract_id                        contract_name  supplier_id  year  \
0         9675          Bioreliance_Master_2018_MSA        11544  2018   
1         9675          Bioreliance_Master_2018_MSA        11544  2019   
2         9675          Bioreliance_Master_2018_MSA        11544  2020   
3         9675          Bioreliance_Master_2018_MSA        11544  2021   
4         9675          Bioreliance_Master_2018_MSA        11544  2022   
5         9675          Bioreliance_Master_2018_MSA        11544  2023   
6         9675          Bioreliance_Master_2018_MSA        11544  2024   
7         9675          Bioreliance_Master_2018_MSA        11544  2025   
8         8157  Kalundborg Bioenergi_Master_2017_SA        17030  2018   
9         8157  Kalundborg Bioenergi_Master_2017_SA        17030  2019   
10        8157  Kalundborg Bioenergi_Master_2017_SA        17030  2020

In [59]:
contract_year.head()

,contract_id,contract_number,contract_name,contract_status,terminated,term_type,start_date,expiration_date,supplier_id,supplier_number,...,Preferred_supplier_tag,start_year,expiry_year,open_ended_contract,end_year,start_year_capped,year,years_to_expiry,contract_age_years,expiry_pressure_bucket
0,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2018,9,0,5y_plus
1,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2019,8,1,5y_plus
2,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2020,7,2,5y_plus
3,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2021,6,3,5y_plus
4,9675,9675,Bioreliance_Master_2018_MSA,published,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,38367,...,0.0,2018,2027,False,2025,2018,2022,5,4,3_to_5y


How many contracts appear each year?

In [60]:
print("\nContracts per year:")
print(contract_year["year"].value_counts().sort_index())


Contracts per year:
year
2015     192
2016     226
2017     262
2018     331
2019     438
2020     578
2021     807
2022    1130
2023    1349
2024    1679
2025    2209
Name: count, dtype: Int64


In [61]:
print("\nRows per contract:")
print(contract_year.groupby("contract_id")["year"].count().describe())


Rows per contract:
count      2209.0
mean     4.165233
std      3.057176
min           1.0
25%           2.0
50%           4.0
75%           6.0
max          11.0
Name: year, dtype: Float64


How many suppliers each year?

In [62]:
contract_year.groupby("year")["supplier_id"].nunique()

year
2015    157
2016    187
2017    207
2018    241
2019    292
2020    314
2021    367
2022    410
2023    450
2024    509
2025    583
Name: supplier_id, dtype: int64

Check for a specific contract as an example:

In [63]:
contract_year.loc[
    contract_year["contract_id"] == "9675",
    [
        "contract_id",
        "contract_name",
        "supplier_id",
        "year",
        "start_year",
        "expiry_year",
        "years_to_expiry",
        "contract_age_years",
        "open_ended_contract",
        "end_year"
    ]
].sort_values("year")

,contract_id,contract_name,supplier_id,year,start_year,expiry_year,years_to_expiry,contract_age_years,open_ended_contract,end_year
0,9675,Bioreliance_Master_2018_MSA,11544,2018,2018,2027,9,0,False,2025
1,9675,Bioreliance_Master_2018_MSA,11544,2019,2018,2027,8,1,False,2025
2,9675,Bioreliance_Master_2018_MSA,11544,2020,2018,2027,7,2,False,2025
3,9675,Bioreliance_Master_2018_MSA,11544,2021,2018,2027,6,3,False,2025
4,9675,Bioreliance_Master_2018_MSA,11544,2022,2018,2027,5,4,False,2025
5,9675,Bioreliance_Master_2018_MSA,11544,2023,2018,2027,4,5,False,2025
6,9675,Bioreliance_Master_2018_MSA,11544,2024,2018,2027,3,6,False,2025
7,9675,Bioreliance_Master_2018_MSA,11544,2025,2018,2027,2,7,False,2025


In [64]:
contract_year.duplicated(subset=["contract_id", "year"]).sum()

np.int64(0)

In [65]:
contract_year[["contract_id", "supplier_id", "year"]].isna().sum()

contract_id    0
supplier_id    0
year           0
dtype: int64

In [66]:
len(contract_year)

9201

In [67]:
contract_year.to_csv(
    "/Users/Thomas/Desktop/Master Thesis/Data/contract_year.csv",
    index=False
)